![QuantConnect Logo](https://cdn.quantconnect.com/web/i/icon.png)
<hr>

## Brain ML Stock Ranking Research

This notebook studies whether Brain ML stock rankings help explain next future returns.

In [ ]:
qb = QuantBook()
# Daily bars will have an end_time that matches the following midnight.
qb.settings.daily_precise_end_time = False

### Build a ML Ranking Universe

Select assets with consistent positive momentum across the 2-, 3-, and 5-day Brain ML horizons, then inspect the returned universe history.

In [ ]:
def select_assets(data: List[BrainStockRankingUniverse]) -> List[Symbol]:
    # Filter for stocks with positive rankings across 2-day, 3-day, and 5-day horizons.
    return [d.symbol for d in data
            if d.rank_2_days and d.rank_2_days > 0.05 and
            d.rank_3_days and d.rank_3_days > 0.05 and
            d.rank_5_days and d.rank_5_days > 0.05]

# Add the Brain ML Stock Ranking universe.
universe = qb.add_universe(BrainStockRankingUniverse, select_assets)
# Request recent universe history.
universe_history = qb.universe_history(universe, qb.time - timedelta(14), qb.time - timedelta(1), flatten=True)
# Print the returned shape and columns.
print(f"Shape: {universe_history.shape}")
print(f"Columns: {list(universe_history.columns)}")
universe_history.head()

### Universe Diagnostics

Check how many assets pass the filter each day and summarize the factors.

In [ ]:
factors = ['rank2days', 'rank3days', 'rank5days']
# Count selected assets by day.
universe_size = universe_history.groupby(level='time').size()
print(f"Universe days: {len(universe_size)}")
# Store the selected symbol list.
unique_assets = list(universe_history.index.levels[1].unique())
print(f"Mean basket size per day: {universe_size.mean():.1f}")
for factor in factors:
    print('')
    print(universe_history[factor].describe())
universe_size.plot(title='Daily Universe Size', ylabel='Universe Size');

### Daily Universe Prices

Fetch daily price history for every symbol that appears in the universe.

In [ ]:
# Get all symbols that appeared in the universe.
symbols = list(universe_history.index.levels[1].unique())
# Request daily price history with one extra day for start-time alignment.
history = qb.history(symbols, universe_history.index[0][0] - timedelta(1), qb.time, Resolution.DAILY)
history

### Align Rankings And Returns

Build a joined table of factors and the future return.

In [ ]:
# Combine the factor values with future returns in a DataFrame.
dataset = universe_history[factors].join(history.open.unstack(0).pct_change().shift(-2).stack().rename('futurereturn').rename_axis(universe_history.index.names)).dropna()
dataset.head()

### Analyze Relationships Between Factor and Future Returns

Create a scatterplot and plot the line of best fit for each factor.

In [ ]:
y = dataset.futurereturn
for factor in factors:    
    # Assign the factor values and future returns.
    x = dataset[factor]
    # Fit a simple linear model.
    slope, intercept = np.polyfit(x, y, 1)
    r_squared = x.corr(y) ** 2
    # Print the linear model statistics.
    print(f"Factor: {factor}")
    print(f"Observations: {len(dataset)}")
    print(f"Mean future return: {y.mean():.2%}")
    print(f"Alpha: {intercept:.2%}")
    print(f"Beta: {slope:.2%}")
    print(f"R-squared: {r_squared:.2%}")
    # Plot the factor values against future returns.
    plt.scatter(x, y, alpha=0.6)
    plt.plot(x.sort_values(), intercept + slope * x.sort_values(), color='tab:red', label='Linear fit')
    plt.axhline(0, color='black', linewidth=1, alpha=0.4)
    plt.title(f'{factor} vs Future Return')
    plt.xlabel(factor)
    plt.ylabel('Future Return')
    plt.legend()
    plt.show()

Create a box plot of the ranking quintiles compared to future returns.

In [ ]:
for factor in factors:
    # Split factor values into quantile buckets.
    x = dataset[factor]
    bucket_count = min(5, x.nunique())
    buckets = pd.qcut(x, q=bucket_count, duplicates='drop')
    # Summarize each bucket with distribution statistics.
    summary = dataset.assign(bucket=buckets).groupby('bucket', observed=True).agg(
        mean_factor=(factor, 'mean'),
        min_future_return=('futurereturn', 'min'),
        max_future_return=('futurereturn', 'max'),
        mean_future_return=('futurereturn', 'mean'),
        std_future_return=('futurereturn', 'std'),
        observations=('futurereturn', 'size')
    ).reset_index()
    summary['bucket'] = summary['bucket'].astype(str)
    # Display the bucket summary.
    print(f"Factor: {factor}")
    display(summary.style.format({
        'mean_factor': '{:.3f}',
        'min_future_return': '{:.2%}',
        'max_future_return': '{:.2%}',
        'mean_future_return': '{:.2%}',
        'std_future_return': '{:.2%}'
    }))
    # Plot the return distribution for each bucket.
    groups = [y[buckets == b].values for b in buckets.cat.categories]
    plt.boxplot(groups, labels=[str(b) for b in buckets.cat.categories])
    plt.axhline(0, color='black', linewidth=1, alpha=0.4)
    plt.title(f'Future Return by {factor} Bucket')
    plt.xlabel(f'{factor} Bucket')
    plt.ylabel('Future Return')
    plt.xticks(rotation=45)
    plt.show()